<a href="https://colab.research.google.com/github/BandaAkshitha/Natural-Language-Processing/blob/main/NLP_Lab15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TASK-1

Import Libraries

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np

Load ELMO Model

In [ ]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Text Corpus

In [ ]:
sentences = ["The bank will not approve the loan",
             "He sat on the river bank"]

Generate Embeddings

In [ ]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']
print(embeddings.shape)

Inspect Word Embedding (“bank”)

In [ ]:
# Convert sentences into tokens
tokenized = [sentence.split() for sentence in sentences]

# Find index of "bank"
idx1 = tokenized[0].index("bank")
idx2 = tokenized[1].index("bank")

# Extract embeddings
bank_emb_1 = embeddings[0][idx1]
bank_emb_2 = embeddings[1][idx2]

print("First 10 values (Sentence 1):", bank_emb_1[:10])
print("First 10 values (Sentence 2):", bank_emb_2[:10])

Compare Context (Cosine Similarity)

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bank_emb_1.numpy(), bank_emb_2.numpy())

print("Similarity between 'bank' meanings:", sim)

TASK-2

BERT model Implementation

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
from torch.nn.functional import cosine_similarity

# Load model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

sentences = [
    "The bank will not approve the loan",
    "He sat on the river bank"
]

embeddings = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors='pt')
    outputs = model(**inputs)

    token_embeddings = outputs.last_hidden_state
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    for i, token in enumerate(tokens):
        if token == 'bank':
            embeddings.append(token_embeddings[0][i])
            break

# Cosine similarity
sim = cosine_similarity(embeddings[0].unsqueeze(0),
                        embeddings[1].unsqueeze(0))

print("Cosine Similarity:", sim.item())

TASK-3

BERT Model Implementation

In [ ]:
sentences = [
    "The bat is flying",
    "He hit the ball with a bat"
]

In [ ]:
# Preprocessing model
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# BERT encoder
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

Preprocess Input

In [ ]:
inputs = preprocess(sentences)
print(inputs.keys())

Generate Embeddings

In [ ]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Get Tokens (Important Step)

In [ ]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)

Extract bat Embeddings

In [ ]:
bat_emb_1 = embeddings[0][2]  # "bat" in first sentence
bat_emb_2 = embeddings[1][7]  # "bat" in second sentence
print("First 10 values (Sentence 1):", bat_emb_1[:10])
print("First 10 values (Sentence 2):", bat_emb_2[:10])

Cosine Similarity

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

TASK-4

ELMO model Implementation

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load ELMo from TensorFlow Hub
elmo = hub.load("https://tfhub.dev/google/elmo/3")

sentences = [
    ["The", "bat", "is", "flying"],
    ["He", "hit", "the", "ball", "with", "a", "bat"]
]

embeddings = elmo.signatures["default"](
    tf.constant([" ".join(s) for s in sentences]) # Pass the joined sentences as a tf.constant directly
)["elmo"]

bat_vectors = []

for i, sentence in enumerate(sentences):
    for j, word in enumerate(sentence):
        if word == "bat":
            bat_vectors.append(embeddings[i][j].numpy())
            break

# Cosine similarity
sim = cosine_similarity([bat_vectors[0]], [bat_vectors[1]])

print("Cosine Similarity:", sim[0][0])

TASK-5

Text Classification Using ELMO+Naive Bayes

Import Libraries

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Text Corpus

In [ ]:
sentences = [
    "Win money now",
    "Claim your prize",
    "Hello how are you",
    "Let's meet tomorrow",
    "Free lottery ticket",
    "Are you coming today"
]

labels = [1, 1, 0, 0, 1, 0]   # 1 = spam, 0 = normal

Load ELMO Model

In [ ]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Generate ELMo Embeddings

In [ ]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']

print("Shape:", embeddings.shape)

Convert to Sentence Embeddings

In [ ]:
sentence_embeddings = tf.reduce_mean(embeddings, axis=1)

X = sentence_embeddings.numpy()
y = np.array(labels)

print("Feature shape:", X.shape)

Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Train Model

In [ ]:
model = GaussianNB()
model.fit(X_train, y_train)

Model Testing

In [ ]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)

Model Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Prediction on New Text

In [ ]:
new_sentence = ["Congratulations! You won a free ticket"]

new_emb = elmo.signatures['default'](tf.constant(new_sentence))['elmo']
new_emb = tf.reduce_mean(new_emb, axis=1)

prediction = model.predict(new_emb.numpy())

print("Prediction:", "Spam" if prediction[0] == 1 else "Not Spam")

TASK-6

Text Classification using BERT+NB

In [ ]:
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

In [ ]:
bert_inputs = preprocess(sentences)

In [ ]:
bert_outputs = bert_model(bert_inputs)

# Use sentence embedding ([CLS])
bert_features = bert_outputs['pooled_output'].numpy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    bert_features, labels, test_size=0.2, random_state=42
)

nb_bert = GaussianNB()
nb_bert.fit(X_train, y_train)

y_pred_bert = nb_bert.predict(X_test)
acc_bert = accuracy_score(y_test, y_pred_bert)

print("BERT Accuracy:", acc_bert)